In [21]:
from dotenv import load_dotenv

load_dotenv()

import sqlite3
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict, Annotated
from langgraph.types import Command
from langchain.chat_models import init_chat_model
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langgraph.checkpoint.sqlite import SqliteSaver

llm = init_chat_model("openai:gpt-4o-mini")
conn = sqlite3.connect(
    "memory.db",
    check_same_thread=False,
)

In [22]:
# class State(TypedDict):
#     messages: Annotated[list[AnyMessage], add_messages]


class State(MessagesState):
    custom_stuff: str


graph_builder = StateGraph(State)

In [23]:
@tool
def get_weather(city: str):
    """Gets weather in city"""
    return f"The weather in {city} is sunny."


llm_with_tools = llm.bind_tools(tools=[get_weather])


def chatbot(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

In [24]:
tool_node = ToolNode(
    tools=[get_weather],
)

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

graph = graph_builder.compile(
    # checkpointer=SqliteSaver(conn),
)

In [25]:
async for event in graph.astream(
    {
        "messages": [
            {
                "role": "user",
                "content": "what is the weather in machupichu",
            }
        ]
    },
    # config={
    #     "configurable": {
    #         "thread_id": "1",
    #     },
    #     "recursion_limit": 50,
    # },
    stream_mode="messages",
):
    print(event)

(AIMessageChunk(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_kZCjSw7uciDdjJ21oW3fhlVb', 'function': {'arguments': '', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={}, id='run--019f0373-1a8f-71e1-aff4-49cc572236df', tool_calls=[{'name': 'get_weather', 'args': {}, 'id': 'call_kZCjSw7uciDdjJ21oW3fhlVb', 'type': 'tool_call'}], tool_call_chunks=[{'name': 'get_weather', 'args': '', 'id': 'call_kZCjSw7uciDdjJ21oW3fhlVb', 'index': 0, 'type': 'tool_call_chunk'}]), {'langgraph_step': 1, 'langgraph_node': 'chatbot', 'langgraph_triggers': ('branch:to:chatbot',), 'langgraph_path': ('__pregel_pull', 'chatbot'), 'langgraph_checkpoint_ns': 'chatbot:4a618cb1-9dca-6ff8-09ae-d44629381bf6', 'checkpoint_ns': 'chatbot:4a618cb1-9dca-6ff8-09ae-d44629381bf6', 'ls_provider': 'openai', 'ls_model_name': 'gpt-4o-mini', 'ls_model_type': 'chat', 'ls_temperature': None})
(AIMessageChunk(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': None, 'function': {'a